<a href="https://colab.research.google.com/github/martinlesch/MAIC1125_M4T1_TAREA_ADRIANA-ZARATE-MARTIN-LESCH/blob/main/MAIC1125_M4T1_TAREA_ADRIANA_ZARATE_%26_MARTIN_LESCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics roboflow

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
import os
from IPython.display import display, Image
from IPython import display
display.clear_output()

In [ ]:
rf = Roboflow(api_key="TU-API-KEY") # Añade tu "API KEY"
project = rf.workspace("martins-workspace-7mzhv").project("maic_m4t3_tarea")
version = project.version(7)
dataset = version.download("yolov8")

In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=30 imgsz=640 batch=16

In [ ]:
!yolo task=detect mode=val model=/content/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

In [ ]:
!yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt source={dataset.location}/valid/images save=True

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob('/content/runs/detect/predict/*.jpg'):
    display(Image(filename=image_path, height=600))
    print ("\n")

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")

metrics = model.val()

print("Precisión (P):        ", metrics.box.mp)
print("Recall (R):           ", metrics.box.mr)
print("mAP50:                ", metrics.box.map50)
print("mAP50-95:             ", metrics.box.map)

In [ ]:
Image(filename='/content/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
Image(filename='/content/runs/detect/train/results.png', width=600)

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage
import os

print('📁 Seleccioná la imagen desde tu computadora...')
uploaded = files.upload()

imagen_path = list(uploaded.keys())[0]
print(f'✅ Imagen cargada: {imagen_path}')

print('\n🖼️ Imagen original:')
display(IPImage(imagen_path, width=600))

In [ ]:
import glob

MODEL_PATH = '/content/runs/detect/train/weights/best.pt'
model = YOLO(MODEL_PATH)

print(f'🔍 Detectando objetos en: {imagen_path}')
results = model.predict(
    source=imagen_path,
    save=True,          # Guarda la imagen con los bounding boxes
    conf=0.25,          # Umbral de confianza (ajustá según necesites)
    iou=0.45,           # Umbral IoU para NMS
    show_labels=True,
    show_conf=True
)

print('\n📊 Resultados de detección:')
for r in results:
    boxes = r.boxes
    if boxes is not None and len(boxes) > 0:
        print(f'  → Objetos detectados: {len(boxes)}')
        for box in boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            label  = model.names[cls_id]
            print(f'     • {label}: {conf:.1%} de confianza')
    else:
        print('  ⚠️ No se detectaron objetos con el umbral actual.')
        print('     Probá bajando conf= a 0.10 o 0.15')

predict_dirs = sorted(glob.glob('/content/runs/detect/predict*'), key=os.path.getmtime)
if predict_dirs:
    ultimo_dir = predict_dirs[-1]
    imagenes_resultado = glob.glob(os.path.join(ultimo_dir, '*.jpg')) + \
                         glob.glob(os.path.join(ultimo_dir, '*.png')) + \
                         glob.glob(os.path.join(ultimo_dir, '*.jpeg'))
    if imagenes_resultado:
        print(f'\n🖼️ Imagen con detecciones (guardada en: {ultimo_dir}):')
        display(IPImage(imagenes_resultado[0], width=600))
    else:
        print('No se encontró imagen de resultado en el directorio.')
else:
    print('No se encontró directorio de predicción.')